In [38]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

import numpy as np
import pandas as pd
from os import listdir
from os.path import isfile, join
from mhcnames import normalize_allele_name
import mhcnames
import mhcgnomes

METHOD = "hmm_pomegranate"

# 1. Read data

In [30]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu

In [36]:
file_name

'data_DRB10101.csv'

In [34]:
onlyfiles = [f for f in listdir(DATA_PATH) if isfile(join(DATA_PATH, f))]
ALLELES = []
NOT_NORMALIZED = []
per_allele_train_df = dict()
for file_name in onlyfiles:
    allele_name = file_name.split("_", 1)[1].split('.')[0]
    NOT_NORMALIZED.append(allele_name)
    allele_name = mhcgnomes.parse(allele_name).to_string()
    per_allele_train_df[allele_name] =  pd.read_csv(DATA_PATH + file_name,sep=";")
    ALLELES.append(allele_name)
ALLELES

['HLA-A*01:01',
 'HLA-A*02:01',
 'HLA-A*03:01',
 'HLA-B*07:02',
 'HLA-B*27:05',
 'HLA-DRB1*01:01']

# 2. Make per length split

In [37]:
from sklearn.model_selection import KFold
MIN_LENGTH = 8
MAX_LENGTH = 30

#leave only binders
def getBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 1]['peptide'].values
    return result_peptides
def getNonBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 0]['peptide'].values
    return result_peptides

def make_per_length_dict(df, binder_flag):
    lengths = set(df['length'])
    lengths = set([length for length in lengths if length >= MIN_LENGTH and length <= MAX_LENGTH])
    per_length_dict = dict()
    for length in lengths:
        result_peptides = df[(df['binder'] == binder_flag) & (df['length'] == length)]['peptide'].values
        per_length_dict[length] = result_peptides
    return per_length_dict

kf = KFold(n_splits=5, shuffle=True)
def make_per_length_per_KFold_dict(per_length_dict):

    per_length_per_kfold_peptides_train = dict()
    per_length_per_kfold_peptides_test = dict()

    print("Length \t Train/Test for splits", end="")
    for target_length in per_length_dict.keys():
        print(f"\n{target_length}:", end="\t")
        peptides_array = per_length_dict[target_length]
        per_length_per_kfold_peptides_train[target_length] = dict()
        per_length_per_kfold_peptides_test[target_length] = dict()
        if len(peptides_array) > kf.n_splits:
            i = 0
            for train_indexes_b, test_indexes_b in kf.split(peptides_array):
                print("{}/{}\t".format(len(train_indexes_b), len(test_indexes_b)), end="")
                current_train = peptides_array[train_indexes_b]
                current_test = peptides_array[test_indexes_b]
                per_length_per_kfold_peptides_train[target_length][i] = current_train
                per_length_per_kfold_peptides_test[target_length][i] = current_test
                i = i + 1
        else:

            print(f"Not enough peptides of length {target_length}", end="")
    print()
    return per_length_per_kfold_peptides_train , per_length_per_kfold_peptides_test





def addAuxColumns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        df['length'] = df['peptide'].str.len()

per_allele_binders_train = dict()
per_allele_non_binders_train = dict()
per_allele_per_length_binders_train = dict()
per_allele_per_length_non_binders_train = dict()
per_allele_per_length_per_kfold_binders_train = dict()
per_allele_per_length_per_kfold_binders_test = dict()
per_allele_per_length_per_kfold_non_binders_train = dict()
per_allele_per_length_per_kfold_non_binders_test = dict()


for allele_name in ALLELES:
    print(f"Allele {allele_name}")
    df_train = per_allele_train_df[allele_name]
    df_train = df_train.sample(frac=1).reset_index(drop=True)
    per_allele_binders_train[allele_name] = getBindersPeptidesOnly(df_train)
    per_allele_non_binders_train[allele_name] = getNonBindersPeptidesOnly(df_train)
    per_allele_per_length_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=1)
    per_allele_per_length_non_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=0)
    print(f"Binders")
    per_allele_per_length_per_kfold_binders_train[allele_name],  per_allele_per_length_per_kfold_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_binders_train[allele_name])
    print(f"Non-binders")
    per_allele_per_length_per_kfold_non_binders_train[allele_name],  per_allele_per_length_per_kfold_non_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_non_binders_train[allele_name])


Allele HLA-A*01:01
Binders
Length 	 Train/Test for splits
8:	764/191	764/191	764/191	764/191	764/191	
9:	5052/1264	5053/1263	5053/1263	5053/1263	5053/1263	
10:	2464/617	2465/616	2465/616	2465/616	2465/616	
11:	1430/358	1430/358	1430/358	1431/357	1431/357	
12:	1106/277	1106/277	1106/277	1107/276	1107/276	
13:	842/211	842/211	842/211	843/210	843/210	
14:	568/142	568/142	568/142	568/142	568/142	
15:	464/117	465/116	465/116	465/116	465/116	
16:	345/87	345/87	346/86	346/86	346/86	
17:	209/53	209/53	210/52	210/52	210/52	
18:	128/33	129/32	129/32	129/32	129/32	
19:	75/19	75/19	75/19	75/19	76/18	
20:	54/14	54/14	54/14	55/13	55/13	
21:	29/8	29/8	30/7	30/7	30/7	
22:	21/6	21/6	22/5	22/5	22/5	
23:	17/5	17/5	18/4	18/4	18/4	
24:	16/4	16/4	16/4	16/4	16/4	
25:	8/2	8/2	8/2	8/2	8/2	
26:	7/2	7/2	7/2	7/2	8/1	
27:	Not enough peptides of length 27
28:	6/2	6/2	6/2	7/1	7/1	
29:	4/2	5/1	5/1	5/1	5/1	
30:	5/2	5/2	6/1	6/1	6/1	
Non-binders
Length 	 Train/Test for splits
8:	7/2	7/2	7/2	7/2	8/1	
9:	2615/654	2615/654

# 3. Save per_length_per_split to files

In [39]:
import os
from collections import defaultdict

TARGET_PATH_TO_FILES = DATA_PATH  + '/per_length_per_kfold_split/'
if not os.path.exists(TARGET_PATH_TO_FILES):
    os.makedirs(TARGET_PATH_TO_FILES)

def save_data(per_allele_per_length_per_kfold_data, data_type=None, dataset_type=None):
    assert data_type in ["binders", "nonbinders"]
    assert dataset_type in ["train", "test"]

    per_allele_dfs = defaultdict(list)
    for allele_name in per_allele_per_length_per_kfold_data.keys():
        for length, per_k_fold_dict in per_allele_per_length_per_kfold_data[allele_name].items():
                for split_num, peptides in per_k_fold_dict.items():
                    df = pd.DataFrame({"peptide":peptides, "length":length, "split": split_num})
                    per_allele_dfs[allele_name].append(df)
        total_df= pd.concat(per_allele_dfs[allele_name])
        target_allele_name = mhcnames.compact_allele_name(allele_name)
        total_df.to_csv(f'{TARGET_PATH_TO_FILES}{target_allele_name}_{data_type}_{dataset_type}.csv',sep=";", index=False)


save_data(per_allele_per_length_per_kfold_binders_train, "binders", "train")
save_data(per_allele_per_length_per_kfold_non_binders_train, "nonbinders", "train")
save_data(per_allele_per_length_per_kfold_binders_test, "binders", "test")
save_data(per_allele_per_length_per_kfold_non_binders_test, "nonbinders", "test")